## GRPO Phase 2 — Nemotron v6

Phase-2 RL training on top of the SFT v6 adapter (`hastws/nemotron-v6-adapter`).
Setup mirrors the working SFT notebook exactly: Triton wheel → Blackwell patch → unsloth + mamba_ssm install → load adapter → GRPO.

Key fixes vs previous version:
- mamba_ssm / causal_conv1d now installed (was the fatal crash)
- adapter loaded via `get_peft_model` + `load_state_dict` (preserves MoE target_parameters)
- dataset from `dgxchen/nemotron-cot-tong` (0.85 LB data), not competition CSV

In [ ]:
# ============================================================================
# §0. Global imports & constants
# ============================================================================
import os, sys, glob, subprocess, site, shutil, stat, json, zipfile
import importlib.util

os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
ADAPTER_PATH    = "/kaggle/input/datasets/hastws/nemotron-v6-adapter"   # hastws/nemotron-v6-adapter dataset (BYOD mount path)
OUTPUT_DIR      = "/kaggle/working/grpo_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GRPO Hyperparameters
GRPO_LR           = 5e-6
GRPO_EPOCHS       = 1
GRPO_NUM_GEN      = 4
GRPO_BATCH        = 1
GRPO_GRAD_ACC     = 4
GRPO_MAX_PROMPT   = 1024
GRPO_MAX_COMPLETE = 1024
SEED              = 42
PROMPT_SUFFIX     = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print("§0 done")

In [ ]:
# ============================================================================
# §1. Install Triton wheel from /kaggle/input into a private target dir.
# ============================================================================
candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")

_TRITON_TARGET = "/kaggle/working/pydeps"
os.makedirs(_TRITON_TARGET, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--no-deps", "--target", _TRITON_TARGET,
     "--upgrade", "--ignore-installed", candidates[0]],
    check=True,
)
if _TRITON_TARGET not in sys.path:
    sys.path.insert(0, _TRITON_TARGET)
site.addsitedir(_TRITON_TARGET)
print(f"Custom target added: {_TRITON_TARGET}")
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
# ============================================================================
# §2. Blackwell (B200) ptxas patch
# ============================================================================
sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'
if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    src_bin = os.path.dirname(ptxas_src)
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
    import triton.backends.nvidia as nv_backend
    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: '12.0'
print('Training environment fixes applied.')

In [ ]:
# ============================================================================
# §3. Offline install of unsloth + trl + mamba_ssm + causal-conv1d wheels
#     (same install sequence as the working SFT notebook)
# ============================================================================
import torch
print(f"Python: {sys.version.split()[0]}  Torch: {torch.__version__}  "
      f"CUDA: {torch.version.cuda}  CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("Training requires a GPU runtime (Nemotron uses CUDA wheels).")

packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
if not os.path.isdir(packages_dir):
    raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--no-index", "--find-links", packages_dir,
     "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes"],
    check=True,
)

all_mamba  = sorted(glob.glob("/kaggle/input/**/mamba_ssm-*.whl",    recursive=True))
all_causal = sorted(glob.glob("/kaggle/input/**/causal*conv1d*.whl", recursive=True))
causal_wheel = all_causal[-1] if all_causal else None
mamba_wheel  = all_mamba[-1]  if all_mamba  else None
print(f"Selected causal wheel: {causal_wheel}")
print(f"Selected mamba wheel:  {mamba_wheel}")
if causal_wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
if mamba_wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
else:
    raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")
print("Offline package installation finished.")

## §4. Load Base Model + SFT v6 Adapter

Load with `FastLanguageModel` (same as SFT notebook), apply identical LoRA config, then load the pretrained adapter weights via `load_state_dict(strict=False)`.
This properly handles both standard LoRA keys **and** the MoE `target_parameters` (gate_up_proj, down_proj) which account for most of the 4GB adapter size.

In [ ]:
# ============================================================================
# §4. Load base model (unsloth) + SFT v6 adapter weights
# ============================================================================
import kagglehub
import torch
from unsloth import FastLanguageModel
from safetensors.torch import load_file

MAX_SEQ_LEN = 4096  # reduced from SFT's 8192; GRPO rollouts need extra VRAM

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print(f"Base model path: {MODEL_PATH}")
print(f"SFT adapter:     {ADAPTER_PATH}")

# --- Load base model (same config as SFT notebook §4) ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Apply LoRA structure (IDENTICAL config to SFT training) ---
# This creates the right parameter layout for both LoRA and MoE target_parameters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj", "up_proj", "down_proj",
    ],
    target_parameters=["mlp.experts.gate_up_proj", "mlp.experts.down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# --- Load pretrained SFT adapter weights ---
# strict=False: frozen base-model keys are correctly absent from the adapter file
adapter_state = load_file(os.path.join(ADAPTER_PATH, "adapter_model.safetensors"))
print(f"\nAdapter file: {len(adapter_state)} tensors  ({sum(v.numel() for v in adapter_state.values())*2/1024**3:.2f} GB)")
print("Sample adapter keys:", list(adapter_state.keys())[:4])

missing, unexpected = model.load_state_dict(adapter_state, strict=False)
n_lora      = sum(1 for k in adapter_state if "lora_" in k)
n_moe       = sum(1 for k in adapter_state if "gate_up_proj" in k or ("down_proj" in k and "lora_" not in k))
print(f"\nLoaded  — LoRA keys: {n_lora}, MoE direct keys: {n_moe}")
print(f"Missing (frozen base): {len(missing)}  |  Unexpected: {len(unexpected)}")
if unexpected:
    print(f"  Unexpected (first 3): {unexpected[:3]}")

model.print_trainable_parameters()
print("✓ v6 SFT adapter loaded — model ready for GRPO")

## §5. GRPO Dataset (0.85 LB cot-tong data)

Use `dgxchen/nemotron-cot-tong` — same data that achieved 0.85 on SFT.
For GRPO we only need `prompt` + `answer`; the `generated_cot` column is discarded.
Stratified sample: ~900 rows across all 6 problem types.

In [ ]:
# ============================================================================
# §5. Build GRPO dataset — same source as 0.85 SFT (nemotron-cot-tong)
# ============================================================================
import re
import pandas as pd
from datasets import Dataset as HFDataset

# Same path as SFT notebook §5
DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"

df = pd.read_csv(DATASET_PATH)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Full dataset: {len(df)} rows   columns: {list(df.columns)}")
if "type" in df.columns:
    print(df["type"].value_counts().to_string())

# GRPO only needs prompt + answer (no generated_cot needed)
# Take up to 800 rows (4 gens × 800 prompts still fits in 9-hr Kaggle limit)
MAX_GRPO_ROWS = 800
grpo_df = df.head(MAX_GRPO_ROWS)
print(f"\nGRPO training set: {len(grpo_df)} samples")

records = []
for _, row in grpo_df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    user_msg = prompt + PROMPT_SUFFIX
    messages = [{"role": "user", "content": user_msg}]
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=True,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
    records.append({"prompt": text, "answer": answer})

grpo_dataset = HFDataset.from_list(records)
print(f"✓ GRPO dataset ready: {len(grpo_dataset)} prompts")
print(f"Example answer: {grpo_dataset[0]['answer']!r}")
print(f"Example prompt (first 250 chars):\n{grpo_dataset[0]['prompt'][:250]}")

## §6. GRPO Training

- `beta=0.0` — no KL penalty, no reference model needed (saves ~30 GB VRAM)
- `temperature=0.7` — lower than 1.0 for higher-quality rollouts
- Reward: `+1.0` correct, `-0.5` boxed but wrong, `-1.0` no boxed answer
- LR 40× lower than SFT to preserve SFT gains while improving RL signal

In [ ]:
# ============================================================================
# §6. GRPO Setup — reward function + GRPOConfig + GRPOTrainer
# ============================================================================
import types as _types

# Mock optional TRL deps that aren't installed (prevents import-time errors)
for _pkg, _attrs in [
    ("mergekit",                  {}),
    ("mergekit.config",           {"MergeConfiguration": type("_M", (), {})}),
    ("mergekit.merge",            {"MergeOptions": type("_M", (), {}), "run_merge": lambda *a, **kw: None}),
    ("mergekit.architecture",     {}),
    ("mergekit.io",               {}),
    ("mergekit.io.tasks",         {}),
    ("mergekit.common",           {}),
    ("llm_blender",               {"Blender": type("_M", (), {})}),
    ("weave",                     {}),
    ("weave.trace",               {}),
    ("weave.trace.context",       {"weave_client_context": type("_M", (), {})}),
    ("liger_kernel",              {}),
    ("liger_kernel.transformers", {}),
]:
    if _pkg not in sys.modules:
        m = _types.ModuleType(_pkg)
        m.__path__ = []
        for k, v in _attrs.items():
            setattr(m, k, v)
        sys.modules[_pkg] = m

from trl import GRPOTrainer, GRPOConfig
print(f"TRL imported successfully")

# --- Reward function ---
_debug_count = 0

def extract_boxed(text):
    """Extract last \\boxed{...} answer, preferring text after </think>."""
    after_think = text[text.rfind("</think>") + 8:] if "</think>" in text else text
    matches = re.findall(r'\\boxed\{([^}]*)\}', after_think)
    if matches:
        return matches[-1].strip()
    # fallback: search full text
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    if matches:
        return matches[-1].strip()
    m = re.search(r'\\boxed\{([^}]*?)$', text)
    return m.group(1).strip() if m else None

def answers_match(pred, gold):
    if pred is None:
        return False
    p, g = pred.strip().lower(), str(gold).strip().lower()
    if p == g:
        return True
    if " ".join(p.split()) == " ".join(g.split()):
        return True
    try:
        pf, gf = float(p), float(g)
        if gf == 0:
            return abs(pf) <= 1e-6
        return abs(pf - gf) / max(abs(gf), 1e-8) <= 1e-3
    except (ValueError, OverflowError):
        pass
    return False

def reward_func(completions, answer, **kwargs):
    global _debug_count
    rewards = []
    for comp, gold in zip(completions, answer):
        text = comp[0]["content"] if isinstance(comp, list) else str(comp)
        pred = extract_boxed(text)
        if _debug_count < 6:
            print(f"[GRPO #{_debug_count}] pred={pred!r} gold={gold!r} tail=...{text[-100:]!r}")
            _debug_count += 1
        if pred is None:
            rewards.append(-1.0)
        elif answers_match(pred, gold):
            rewards.append(1.0)
        else:
            rewards.append(-0.5)
    return rewards

# GRPOTrainer compatibility patch
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

grpo_config = GRPOConfig(
    output_dir="/kaggle/working/grpo_ckpt",
    num_train_epochs=GRPO_EPOCHS,
    per_device_train_batch_size=GRPO_BATCH,
    gradient_accumulation_steps=GRPO_GRAD_ACC,
    learning_rate=GRPO_LR,
    num_generations=GRPO_NUM_GEN,
    max_completion_length=GRPO_MAX_COMPLETE,
    max_prompt_length=GRPO_MAX_PROMPT,
    temperature=0.7,
    beta=0.0,                    # no KL penalty → no reference model needed
    bf16=True,
    max_grad_norm=1.0,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    seed=SEED,
)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_func],
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
)

est_steps = len(grpo_dataset) * GRPO_EPOCHS // (GRPO_BATCH * GRPO_GRAD_ACC)
print(f"\nGRPO ready:")
print(f"  Dataset: {len(grpo_dataset)} samples × {GRPO_NUM_GEN} gens  →  ~{est_steps} gradient steps")
print(f"  LR={GRPO_LR}, max_complete={GRPO_MAX_COMPLETE}, beta=0.0, temperature=0.7")

In [ ]:
import time

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory before training: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

print("Starting GRPO training...")
t0 = time.time()
grpo_trainer.train()
elapsed = (time.time() - t0) / 60
print(f"\n✓ GRPO training done in {elapsed:.1f} min")

if torch.cuda.is_available():
    torch.cuda.synchronize()
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

## §7. Save GRPO Adapter + Package submission.zip

In [ ]:
# ============================================================================
# §7. Save GRPO adapter + package submission.zip
# ============================================================================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:45s}  {size/1024/1024:8.1f} MB")

# Package into submission.zip (competition format: adapter files at root)
zip_path = "/kaggle/working/submission.zip"
required_files = ["adapter_config.json", "adapter_model.safetensors"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing {fname} in {OUTPUT_DIR}")
        zf.write(fpath, fname)
        print(f"  Added {fname}")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
print("✓ submission.zip ready for competition submission")